# Chapter 6 — Kernel Methods and Feature Spaces

**Math for ML Engineers** · companion notebook

This notebook mirrors §6.2 (the RBF/Gaussian kernel) and §6.4 (the GP posterior) of the chapter, walking from the kernel function itself to a full Gaussian Process regression fit. **Key takeaway: a kernel lets you work in an implicit, possibly infinite-dimensional feature space without ever computing a feature vector — the entire GP posterior below is built from nothing but pairwise kernel evaluations.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(6)

## The RBF kernel and the GP prior

A kernel $K(x, x')$ is a similarity function that implicitly computes an inner product in some (possibly infinite-dimensional) feature space $\phi$:

$$
K(x, x') = \phi(x) \cdot \phi(x')
$$

The **radial basis function (RBF)**, a.k.a. squared-exponential or Gaussian kernel, is the workhorse choice (§6.2):

$$
K(x, x') = \sigma_f^2 \exp\!\left(-\frac{\lVert x - x' \rVert^2}{2\ell^2}\right)
$$

where $\ell$ (`length_scale`) controls how quickly correlation decays with distance, and $\sigma_f^2$ (`signal_var`) scales the overall output variance. A **Gaussian Process** places a distribution directly over functions $f$ by declaring that any finite collection of function values is jointly Gaussian:

$$
f(x_1), \dots, f(x_n) \sim \mathcal{N}(0, K)
$$

with $K_{ij} = K(x_i, x_j)$. GP regression is Bayesian inference on this prior in closed form — no gradient descent required.

In [ ]:
def rbf_kernel(X1, X2, length_scale, signal_var):
    # K(x,x') = signal_var * exp(-||x-x'||^2 / (2 * l^2))
    X1 = np.atleast_1d(X1)
    X2 = np.atleast_1d(X2)
    sq_dists = (X1[:, None] - X2[None, :]) ** 2
    return signal_var * np.exp(-sq_dists / (2 * length_scale ** 2))


# True function and noisy training data: sin(x) over [-3*pi, 3*pi]
N_TRAIN = 15
NOISE_STD = 0.15

X_train = np.sort(np.random.uniform(-3 * np.pi, 3 * np.pi, N_TRAIN))
y_train = np.sin(X_train) + NOISE_STD * np.random.randn(N_TRAIN)

X_test = np.linspace(-3 * np.pi, 3 * np.pi, 200)
y_true = np.sin(X_test)

## GP posterior via Cholesky (§6.4)

Given training inputs $X$, noisy targets $y = f(X) + \varepsilon$ with $\varepsilon \sim \mathcal{N}(0, \sigma_n^2 I)$, and test inputs $X_*$, the joint Gaussian prior over $(y, f_*)$ conditions down to a closed-form posterior:

$$
\mu_* = K_{*n}(K_{nn} + \sigma_n^2 I)^{-1} y, \qquad
\Sigma_* = K_{**} - K_{*n}(K_{nn} + \sigma_n^2 I)^{-1} K_{n*}
$$

Directly inverting $K_{nn} + \sigma_n^2 I$ is numerically fragile and $O(n^3)$ regardless, so we instead factor $K_{nn} + \sigma_n^2 I = LL^\top$ (Cholesky) and solve two triangular systems — this is the standard "GP via Cholesky" recipe: stable, and it reuses $L$ for both the mean and the diagonal of the predictive variance.

In [ ]:
def gp_posterior(X_train, y_train, X_test, kernel_fn, noise_var):
    K_nn = kernel_fn(X_train, X_train) + noise_var * np.eye(len(X_train))
    K_star_n = kernel_fn(X_test, X_train)
    K_star_star = kernel_fn(X_test, X_test)

    L = np.linalg.cholesky(K_nn)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y_train))
    mu_star = K_star_n @ alpha

    v = np.linalg.solve(L, K_star_n.T)
    sigma_star = np.sqrt(np.maximum(np.diag(K_star_star) - np.sum(v ** 2, axis=0), 0))

    return mu_star, sigma_star


LENGTH_SCALE = 1.0
SIGNAL_VAR = 1.0
NOISE_VAR = NOISE_STD ** 2

kernel_fn = lambda A, B: rbf_kernel(A, B, LENGTH_SCALE, SIGNAL_VAR)
mu_star, sigma_star = gp_posterior(X_train, y_train, X_test, kernel_fn, NOISE_VAR)

rmse = np.sqrt(np.mean((mu_star - y_true) ** 2))
coverage = np.mean(np.abs(y_true - mu_star) <= 2 * sigma_star)

print(f"RMSE (posterior mean vs true sin(x)): {rmse:.4f}")
print(f"95% coverage (fraction within +/-2 sigma): {coverage:.3f}")

## Visualizing the fit

The classic "GP fan plot": training points, the true underlying function, the posterior mean, and a shaded $\pm 2\sigma$ band that should (by construction) contain the true curve about 95% of the time. Then a 3-panel sweep over `length_scale` $\in \{0.3, 1.0, 3.0\}$ shows the smoothness/overfitting tradeoff directly: too short and the posterior chases noise between points; too long and it oversmooths, flattening genuine structure in $\sin(x)$.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(X_test, y_true, "k--", label="true sin(x)")
ax.plot(X_test, mu_star, "C0", label="posterior mean")
ax.fill_between(X_test, mu_star - 2 * sigma_star, mu_star + 2 * sigma_star,
                 color="C0", alpha=0.2, label=r"$\pm 2\sigma$ band")
ax.scatter(X_train, y_train, color="C3", zorder=5, label="training points")
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.set_title(f"GP posterior fit (length_scale={LENGTH_SCALE})")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

print(f"Posterior mean tracks sin(x) with RMSE={rmse:.4f}; "
      f"{coverage:.1%} of true values fall inside the +/-2 sigma band "
      "(target ~95% for a well-calibrated posterior).")

# --- length_scale sweep: smoothness/overfitting tradeoff ---
length_scales = [0.3, 1.0, 3.0]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

for ax, ell in zip(axes, length_scales):
    k_fn = lambda A, B, ell=ell: rbf_kernel(A, B, ell, SIGNAL_VAR)
    mu_ell, sigma_ell = gp_posterior(X_train, y_train, X_test, k_fn, NOISE_VAR)
    rmse_ell = np.sqrt(np.mean((mu_ell - y_true) ** 2))

    ax.plot(X_test, y_true, "k--", linewidth=1, label="true sin(x)")
    ax.plot(X_test, mu_ell, "C0", label="posterior mean")
    ax.fill_between(X_test, mu_ell - 2 * sigma_ell, mu_ell + 2 * sigma_ell,
                     color="C0", alpha=0.2)
    ax.scatter(X_train, y_train, color="C3", s=20, zorder=5)
    ax.set_title(f"length_scale={ell}\nRMSE={rmse_ell:.3f}")
    ax.set_xlabel("x")

axes[0].set_ylabel("f(x)")
axes[0].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

print("Small length_scale -> wiggly posterior that chases noise (locally overfit, "
      "wide uncertainty away from data). Large length_scale -> oversmoothed posterior "
      "that underfits sin(x)'s curvature. length_scale=1.0 sits closest to the true "
      "correlation structure of sin(x) here.")

## From kernels to attention

The kernel trick's real content is this: $K(x, x') = \phi(x) \cdot \phi(x')$ lets you compute similarity in a feature space you never materialize. The RBF kernel above corresponds to an *infinite*-dimensional $\phi$ — yet every quantity we needed (`K_nn`, `K_star_n`, `K_star_star`) was just pairwise evaluations of a similarity function.

This is exactly the mathematical move inside **scaled dot-product attention**. The attention weights $\text{softmax}(QK^\top / \sqrt{d_k})$ are a *bounded, learned similarity kernel* between query and key vectors — $QK^\top$ is literally a Gram matrix of inner products (a linear kernel on learned projections), and softmax turns it into a normalized weighting, the same role $K_{*n}(K_{nn}+\sigma_n^2 I)^{-1}$ plays in the GP posterior mean: a similarity-weighted combination of observed values. Every transformer forward pass is, at each layer, computing a kernel-weighted average.

The other production payoff of this chapter's machinery is **Bayesian optimization**. The $\pm 2\sigma$ band in the fan plot above isn't decoration — GP-based Bayesian opt uses exactly that posterior uncertainty as an acquisition signal (e.g. upper confidence bound or expected improvement) to decide where to sample next when tuning hyperparameters of an expensive-to-train model. Wide $\sigma_*$ means "unexplored, worth trying"; this is how frameworks like Ax/BoTorch and Vizier drive learning-rate, batch-size, and architecture search for large training runs where every trial costs real GPU-hours.

In [ ]:
# Toy Bayesian-optimization-style acquisition: pick the next x to sample using
# an Upper Confidence Bound acquisition on the GP posterior itself.
kappa = 2.0  # exploration weight, same 2-sigma logic as the fan plot band
ucb = mu_star + kappa * sigma_star
next_x_idx = np.argmax(ucb)
next_x = X_test[next_x_idx]

print(f"Next point to sample (max UCB, kappa={kappa}): x = {next_x:.3f}")
print(f"  posterior mean there: {mu_star[next_x_idx]:.3f}, "
      f"posterior std there: {sigma_star[next_x_idx]:.3f}")

# Show that UCB-selected points concentrate in high-uncertainty (data-sparse) regions,
# not just high-mean regions -- the exploration/exploitation split acquisition functions rely on.
dist_to_nearest_train = np.min(np.abs(next_x - X_train))
mean_dist_to_nearest_train = np.mean([np.min(np.abs(x - X_train)) for x in X_test])
print(f"\nDistance from UCB-selected x to nearest training point: "
      f"{dist_to_nearest_train:.3f} (grid-average nearest-neighbor distance: "
      f"{mean_dist_to_nearest_train:.3f})")
print("UCB is favoring a data-sparse region over the grid average: "
      f"{dist_to_nearest_train > mean_dist_to_nearest_train}")

print(f"\nMean posterior std overall: {sigma_star.mean():.4f}; "
      f"std at UCB-selected point: {sigma_star[next_x_idx]:.4f} "
      f"({sigma_star[next_x_idx] / sigma_star.mean():.2f}x the average -- "
      "acquisition is actively seeking uncertain regions, exactly as Bayesian "
      "optimization exploits the GP's calibrated uncertainty band.)")

## Exercise

1. The RBF kernel is one member of the Matérn family (as $\nu \to \infty$). Derive the Matérn-3/2 kernel's form and modify `rbf_kernel` into `matern32_kernel`; refit the GP on the same `X_train, y_train` and compare RMSE and 95% coverage. §6.2 gives the closure rules for building valid alternative covariance functions if you need the closed form.

2. The attention analogy in the insight cell above treats $QK^\top$ as an (unnormalized, linear) kernel Gram matrix. Work out precisely what feature map $\phi$ would have to be, for a single attention head, to make $\text{softmax}(QK^\top/\sqrt{d_k})$ a properly normalized kernel in the Mercer sense — and where the analogy breaks down (hint: Mercer kernels must be symmetric and PSD; is $QK^\top$ either, in general?). §6.2 (PSD kernels and Mercer's theorem) is the relevant background.